In [1]:
import pandas as pd
import json
import os

In [2]:
#  2) 통계 계산 함수
def compute_stats(df):
    total = len(df)
    normal = int((df['status']=='정상').sum())
    broken = int((df['status']=='고장').sum())
    year_counts = df['install_year'].value_counts().sort_index().to_dict()
    fault_rate = {
        int(yr): round(group['status'].eq('고장').mean()*100,1)
        for yr, group in df.groupby('install_year')
    }
    return {
        'total': total,
        'normal': normal,
        'broken': broken,
        'year_counts': year_counts,
        'fault_rate': fault_rate
    }

In [15]:
csv_path = os.path.join('../','static', 'data', 'streetlights_data3.csv')
df = pd.read_csv(csv_path)
# 3) JSON 파일로 저장
stats = compute_stats(df)

# dong_state 넣기
dong_df = pd.read_csv(os.path.join('../','static', 'data', 'dong_stats_no_geometry.csv'), encoding='utf-8')
# dict 형태로 변환
# dong_df.to_dict(orient='records')
stats['dong_state'] = dong_df.to_dict(orient='records')

In [16]:
out_path = os.path.join('../','static','data','streetlights_stats.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"Saved stats to {out_path}")

Saved stats to ../static/data/streetlights_stats.json


---

In [47]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import wkt
from shapely.geometry import Polygon


fp = "/home/nute11a/workspace/PBL/SUWON/code/data/origin/LARD_ADM_SECT_SGG_41_202505.shp"
gdf = gpd.read_file(fp, engine="pyogrio")
print("Original CRS:", gdf.crs)

mask = gdf['SGG_NM'] == '경기도 수원시 영통구'
gdf = gdf[mask]

# ── (2) Meter 단위 연산을 위해 Web Mercator 로 투영 ───────────────
# EPSG:3857 은 미터 단위이므로 거리를 쉽게 계산할 수 있습니다.
gdf_m = gdf.to_crs(epsg=3857)

# ── (3) 격자(Cell) 생성 파라미터 ─────────────────────────────────
cell_size = 100  # 미터 단위 격자 크기
minx, miny, maxx, maxy = gdf_m.total_bounds

# x, y 축으로 일정 간격의 좌표를 생성
xs = np.arange(minx, maxx, cell_size)
ys = np.arange(miny, maxy, cell_size)

# 셀(Polygon) 리스트 생성
cells = []
for x in xs:
    for y in ys:
        cells.append(Polygon([
            (x,       y),
            (x + cell_size, y),
            (x + cell_size, y + cell_size),
            (x,       y + cell_size)
        ]))

# ── (4) Grid GeoDataFrame 생성 및 경계와 교집합 ────────────────────
grid = gpd.GeoDataFrame({"geometry": cells}, crs="EPSG:3857")
# 경계 내부만 남기기
grid_clipped = gpd.overlay(grid, gdf_m, how="intersection")

# ── (5) 결과를 WGS84(4326)로 복원 ────────────────────────────────
grid_final = grid_clipped.to_crs(epsg=4326)
gdf_wgs84  = gdf.to_crs(epsg=4326)

print(grid_final.head())
print("Grid count:", len(grid_final))
print("Final CRS:", grid_final.crs)

grid_final.to_file("suwon_100m_grid.geojson", driver="GeoJSON")

Original CRS: EPSG:5186
  ADM_SECT_C       SGG_NM  SGG_OID COL_ADM_SE  \
0      41117  경기도 수원시 영통구   3618.0      41110   
1      41117  경기도 수원시 영통구   3618.0      41110   
2      41117  경기도 수원시 영통구   3618.0      41110   
3      41117  경기도 수원시 영통구   3618.0      41110   
4      41117  경기도 수원시 영통구   3618.0      41110   

                                            geometry  
0  POLYGON ((127.03329 37.26701, 127.03329 37.266...  
1  POLYGON ((127.03329 37.26701, 127.03318 37.267...  
2  POLYGON ((127.03329 37.29989, 127.03329 37.299...  
3  POLYGON ((127.03329 37.30060, 127.03329 37.299...  
4  POLYGON ((127.03329 37.30132, 127.03329 37.300...  
Grid count: 4680
Final CRS: EPSG:4326


In [62]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import wkt
from shapely.geometry import Polygon


fp = "/home/nute11a/workspace/PBL/SUWON/code/data/origin/LARD_ADM_SECT_SGG_41_202505.shp"
gdf = gpd.read_file(fp, engine="pyogrio")
print("Original CRS:", gdf.crs)

mask = gdf['SGG_NM'] == '경기도 수원시 영통구'
gdf = gdf[mask]

# ── (2) Meter 단위 연산을 위해 Web Mercator 로 투영 ───────────────
# EPSG:3857 은 미터 단위이므로 거리를 쉽게 계산할 수 있습니다.
gdf_m = gdf.to_crs(epsg=3857)

# ── (3) 격자(Cell) 생성 파라미터 ─────────────────────────────────
cell_size = 100  # 미터 단위 격자 크기
minx, miny, maxx, maxy = gdf_m.total_bounds

# x, y 축으로 일정 간격의 좌표를 생성
xs = np.arange(minx, maxx, cell_size)
ys = np.arange(miny, maxy, cell_size)

# 셀(Polygon) 리스트 생성
cells = []
for x in xs:
    for y in ys:
        cells.append(Polygon([
            (x,       y),
            (x + cell_size, y),
            (x + cell_size, y + cell_size),
            (x,       y + cell_size)
        ]))

# ── (4) Grid GeoDataFrame 생성 및 경계와 교집합 ────────────────────
grid = gpd.GeoDataFrame({"geometry": cells}, crs="EPSG:3857")
# 경계 내부만 남기기
grid_clipped = gpd.overlay(grid, gdf_m, how="intersection")

# ── (5) 결과를 WGS84(4326)로 복원 ────────────────────────────────
grid_final = grid_clipped.to_crs(epsg=4326)
gdf_wgs84  = gdf.to_crs(epsg=4326)

print(grid_final.head())
print("Grid count:", len(grid_final))
print("Final CRS:", grid_final.crs)
grid_final = grid_final.reset_index().rename(columns={"index":"grid_id"})
grid_final.to_file("suwon_100m_grid.geojson", driver="GeoJSON")

Original CRS: EPSG:5186
  ADM_SECT_C       SGG_NM  SGG_OID COL_ADM_SE  \
0      41117  경기도 수원시 영통구   3618.0      41110   
1      41117  경기도 수원시 영통구   3618.0      41110   
2      41117  경기도 수원시 영통구   3618.0      41110   
3      41117  경기도 수원시 영통구   3618.0      41110   
4      41117  경기도 수원시 영통구   3618.0      41110   

                                            geometry  
0  POLYGON ((127.03329 37.26701, 127.03329 37.266...  
1  POLYGON ((127.03329 37.26701, 127.03318 37.267...  
2  POLYGON ((127.03329 37.29989, 127.03329 37.299...  
3  POLYGON ((127.03329 37.30060, 127.03329 37.299...  
4  POLYGON ((127.03329 37.30132, 127.03329 37.300...  
Grid count: 4680
Final CRS: EPSG:4326


In [74]:
from shapely.geometry import Point


grid = gpd.read_file('/home/nute11a/workspace/PBL/SUWON/code/data/preprocessed/suwon_100m_grid.geojson', engine="pyogrio")
grid = grid.reset_index().rename(columns={"index":"grid_id"})
df_lamps  = pd.read_csv('/home/nute11a/workspace/PBL/SUWON/code/data/preprocessed/streetlights_data3.csv')
geometry = [Point(xy) for xy in zip(df_lamps['lon'], df_lamps['lat'])]
lamps = gpd.GeoDataFrame(df_lamps, geometry=geometry, crs="EPSG:4326")

joined = gpd.sjoin(lamps, grid[['grid_id','geometry']], how="inner", predicate="within")

# 6) 격자별 가로등 개수 집계
counts = joined.groupby("grid_id").size().rename("lamp_count")

# 7) grid에 lamp_count 합치기 (없는 격자는 0으로)
grid = grid.set_index("grid_id")
grid["lamp_count"] = counts
grid["lamp_count"] = grid["lamp_count"].fillna(0).astype(int)
grid = grid.reset_index()

# 8) 결과 확인
print(grid[["grid_id","lamp_count"]].head())

# ── (9) 램프 밀도 계산 및 시그모이드 매핑 ───────────────────────────
grid_m = grid.to_crs(epsg=3857)
# 3) 셀 면적(km²) → 셀 한 변 길이(km) 계산
grid_m["area_km2"] = grid_m.geometry.area / 1e6
grid_m["cell_km"]   = np.sqrt(grid_m["area_km2"])

# 4) 램프 밀도 계산 (개수 / km)
grid_m["density"] = grid_m["lamp_count"] / grid_m["cell_km"]

# 5) 밀도 정규화 (0~1)
d_min, d_max = grid_m["density"].min(), grid_m["density"].max()
grid_m["d_norm"] = (grid_m["density"] - d_min) / (d_max - d_min + 1e-9)

# 6) 시그모이드 매핑으로 score 생성
def sigmoid(x, x0=0.3, k=10):
    return 1 / (1 + np.exp(-k * (x - x0)))

grid_m["score"] = sigmoid(grid_m["d_norm"], x0=0.3, k=10)

# 7) WGS84 복원 및 저장
grid_scored = grid_m.to_crs(epsg=4326)
grid_scored.to_file("data/preprocessed/grid_with_score.geojson", driver="GeoJSON")

grid_scored.head()

   grid_id  lamp_count
0        0           0
1        1           0
2        2           0
3        3           0
4        4           0


,grid_id,ADM_SECT_C,SGG_NM,SGG_OID,COL_ADM_SE,geometry,lamp_count,area_km2,cell_km,density,d_norm,score
0,0,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.26701, 127.03329 37.266...",0,0.000044,0.006626,0.0,0.0,0.047426
1,1,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.26701, 127.03318 37.267...",0,0.000155,0.012437,0.0,0.0,0.047426
2,2,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.29989, 127.03329 37.299...",0,0.001854,0.043063,0.0,0.0,0.047426
3,3,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.30060, 127.03329 37.299...",0,0.005738,0.075748,0.0,0.0,0.047426
4,4,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.30132, 127.03329 37.300...",0,0.005841,0.076425,0.0,0.0,0.047426


In [71]:
grid

,grid_id,ADM_SECT_C,SGG_NM,SGG_OID,COL_ADM_SE,geometry,lamp_count
0,0,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.26701, 127.03329 37.266...",0
1,1,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.26701, 127.03318 37.267...",0
2,2,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.29989, 127.03329 37.299...",0
3,3,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.30060, 127.03329 37.299...",0
4,4,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.03329 37.30132, 127.03329 37.300...",0
...,...,...,...,...,...,...,...
4675,4675,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.08898 37.29488, 127.08898 37.295...",0
4676,4676,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.08898 37.29560, 127.08898 37.296...",0
4677,4677,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.08898 37.29703, 127.08907 37.297...",0
4678,4678,41117,경기도 수원시 영통구,3618.0,41110,"POLYGON ((127.08898 37.29703, 127.08898 37.297...",0
